# Lezione 07 - Design Pattern di Pianificazione

Questo notebook dimostra il **Design Pattern di Pianificazione** per agenti AI utilizzando il Microsoft Agent Framework.
Imparerai come suddividere richieste di viaggio complesse in sotto-compiti strutturati, assegnarli ad agenti specialistici,
ed eseguire il piano risultante — tutto utilizzando output strutturato basato su modelli Pydantic.


## Configurazione


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os, asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Scomposizione del Compito

La scomposizione del compito è il nucleo del pattern di progettazione della pianificazione. Invece di chiedere a un singolo agente di
gestire una richiesta complessa dall'inizio alla fine, suddividiamo il problema in **sottocompiti** più piccoli e ben definiti.
Ogni sottocompito viene assegnato a un agente specialista (ad esempio, voli, hotel, attività) con chiare
priorità e ordinamento delle dipendenze.

Questo approccio offre diversi vantaggi:
- **Chiarezza**: ogni sottocompito ha una singola responsabilità.
- **Parallelismo**: i sottocompiti indipendenti possono essere eseguiti contemporaneamente.
- **Affidabilità**: i fallimenti sono isolati ai singoli sottocompiti.
- **Tracciamento del budget**: i costi sono stimati per sottocompito e sommati.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## Creare un agente di pianificazione con output strutturato

L'agente di pianificazione agisce come un **coordinatore alla reception**. Data una richiesta di viaggio di alto livello,
produce un `TravelPlan` strutturato — scomponendo la richiesta in sottoattività, stabilendo le priorità,
e identificando le dipendenze in modo che un concierge o un livello di esecuzione possano svolgere il lavoro.


In [ ]:
planning_agent = client.as_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    options={"response_format": TravelPlan}
)
if result:
    plan = result.value
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## Esecuzione di un Piano con Strumenti Specializzati

Una volta che l'agente alla reception ha prodotto un piano strutturato, lo esegue il **concierge agent**.
Ogni strumento specializzato gestisce una categoria di sottoattività (voli, hotel, attività). Il concierge
procede attraverso le sottoattività del piano in ordine di dipendenza e assegna ciascuna allo
strumento appropriato.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = client.as_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## Sommario

In questa lezione hai appreso il **Pattern di Progettazione della Pianificazione** per agenti AI:

1. **Decomposizione del Compito** — Un agente di pianificazione alla reception suddivide una richiesta di viaggio complessa in
   sottocompiti strutturati utilizzando modelli Pydantic, assegnando ciascuno a un agente specialista con priorità
   e dipendenze.
2. **Output Strutturato** — Passando un `response_format` l'agente restituisce un oggetto `TravelPlan` validato
   invece di testo libero, rendendo l'elaborazione successiva affidabile.
3. **Esecuzione del Piano** — Un agente concierge itera attraverso i sottocompiti usando strumenti specialistici
   (`book_flight`, `reserve_hotel`, `book_activity`) per realizzare il piano e riportare i risultati.

Questo pattern separa *cosa fare* (pianificazione) da *come farlo* (esecuzione), rendendo gli agenti
più modulari, testabili e più facili da estendere.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Questo documento è stato tradotto utilizzando il servizio di traduzione AI [Co-op Translator](https://github.com/Azure/co-op-translator). Sebbene ci impegniamo per garantire la precisione, si prega di notare che le traduzioni automatizzate possono contenere errori o imprecisioni. Il documento originale nella sua lingua nativa deve essere considerato la fonte autorevole. Per informazioni critiche, si raccomanda una traduzione professionale effettuata da un essere umano. Non siamo responsabili per eventuali malintesi o interpretazioni errate derivanti dall’uso di questa traduzione.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
